# Benchmark 1: 2 Class vs 4 Class: Cross-Session

In [1]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score
from pyriemann.estimation import Covariances
import scipy

In [2]:
from pyriemann.utils import mean_riemann
from scipy.optimize import minimize
from pymanopt import Problem
from pymanopt.manifolds import SpecialOrthogonalGroup
from pymanopt.optimizers import SteepestDescent
from pymanopt import Problem
from functools import partial
from pymanopt.function import numpy as pymanopt_numpy
import autograd.numpy as anp  # Autograd's NumPy replacement
from autograd import grad
import pymanopt
import autograd.scipy.linalg as linalg
from pyriemann.classification import MDM

In [3]:
def logm_approx(A):
    I = anp.eye(A.shape[0])  # Identity matrix
    return A - I - 0.5 * (A - I) @ (A - I) 

def frobenius_norm(X):
    return anp.sqrt(anp.sum(X**2))


In [4]:
def encode_labels(labels_list):
    encoded_list = []
    for labels in labels_list:
        # Create mapping from original labels to 0,1
        unique_labels = np.unique(labels)
        label_map = {unique_labels[i]: i for i in range(len(unique_labels))}
        
        # Apply mapping
        encoded = np.array([label_map[label] for label in labels])
        encoded_list.append(encoded)
    return encoded_list

In [5]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [6]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.

tmin = 0.5       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = 3.5  # This gives 4.0 seconds.


In [7]:
preproc_options = {
        'None': None,
        'Standard': (8, 30),
        'Alpha': (8, 12),
        'Beta': (13, 30)
    }

In [8]:
def twofour_crosssession(n_classes):
    for proc_name, freq_range in preproc_options.items():
        data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCI2b/BCICIV_2b_gdf'

        # Lists to hold data for all subjects
        train_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
        train_active_y = []         # List to hold event labels per subject
        train_active_metadata = []  # List to hold event metadata per subject

        # Define subject IDs (B01 to B09)
        subjects = [f'B{subj:02d}' for subj in range(1, 10)]

        for subj in subjects:
            # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
            session_ids = ['01T'] #, '02T', '03T']
            subj_epochs_list = []

            for sess in session_ids:
                filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
                
                # Check if file exists to avoid errors
                if not os.path.exists(filename):
                    print(f"File {filename} not found, skipping.")
                    continue
                
                # Load the GDF file
                raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
                
                # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
                event_id_mapping = {'769': 1, '770': 2}
                events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
                
                # Select only EEG channels (C3, Cz, C4)
                print(raw.ch_names)
                eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
                if len(eeg_channels) != 3:
                    print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
                raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
                
                # Create epochs
                epochs = mne.Epochs(
                    raw_eeg,
                    events,
                    event_id={'left': 1, 'right': 2},
                    tmin=tmin,
                    tmax=tmax,
                    baseline=None,  # No baseline correction, matching your 2a code
                    preload=True,
                    verbose=False
                )
                
                subj_epochs_list.append(epochs)
            
            # Skip subject if no sessions were processed
            if not subj_epochs_list:
                print(f"No valid sessions found for subject {subj}, skipping.")
                continue
            
            # Concatenate epochs across sessions for this subject
            if len(subj_epochs_list) > 1:
                subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
            else:
                subj_epochs = subj_epochs_list[0]
            
            # Get the epoch data
            subj_data = subj_epochs.get_data()
            
            # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
            n_trials, n_channels, n_times = subj_data.shape
            fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
            subj_filtered_data = np.empty_like(subj_data)
        
            if freq_range is None:
                subj_filtered_data = subj_data.copy()
            else:
                lowcut, highcut = freq_range
                for trial in range(n_trials):
                    for ch in range(n_channels):
                        subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                            subj_data[trial, ch, :],
                            lowcut=lowcut,    # Lower bound of sensorimotor rhythm.
                            highcut=highcut,  # Upper bound of sensorimotor rhythm.
                            fs=fs,
                            order=50     # Lower order for a smoother causal filter.
                        )
            # Append processed data, labels, and metadata
            train_active_X.append(subj_filtered_data)
            train_active_y.append(subj_epochs.events[:, 2])  # Labels in third column (1 or 2)
            train_active_metadata.append(subj_epochs.events)
            
            # Print shape to verify
            print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

        print(f"Loaded data for {len(train_active_X)} subjects.")


        # Lists to hold data for all subjects
        eval_active_X = []         # List to hold numpy arrays with shape (n_trials, n_channels, n_times) per subject
        eval_active_y = []         # List to hold event labels per subject
        eval_active_metadata = []  # List to hold event metadata per subject

        # Define subject IDs (B01 to B09)
        subjects = [f'B{subj:02d}' for subj in range(1, 10)]

        for subj in subjects:
            # Define training sessions for this subject (e.g., B0101T.gdf, B0102T.gdf, B0103T.gdf)
            session_ids = ['02T'] #, '02T', '03T']
            subj_epochs_list = []

            for sess in session_ids:
                filename = os.path.join(data_dir, f'{subj}{sess}.gdf')
                
                # Check if file exists to avoid errors
                if not os.path.exists(filename):
                    print(f"File {filename} not found, skipping.")
                    continue
                
                # Load the GDF file
                raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)
                
                # Extract events from annotations, mapping '769' to 1 (left) and '770' to 2 (right)
                event_id_mapping = {'769': 1, '770': 2}
                events, event_dict = mne.events_from_annotations(raw, event_id=event_id_mapping, verbose=False)
                
                # Select only EEG channels (C3, Cz, C4)
                print(raw.ch_names)
                eeg_channels = [ch for ch in raw.ch_names if ch in ['EEG:C3', 'EEG:Cz', 'EEG:C4']]
                if len(eeg_channels) != 3:
                    print(f"Warning: Expected 3 EEG channels for {subj}{sess}, found {len(eeg_channels)}: {eeg_channels}")
                raw_eeg = raw.pick_channels(eeg_channels, verbose=False)
                
                
                # Create epochs
                epochs = mne.Epochs(
                    raw_eeg,
                    events,
                    event_id={'left': 1, 'right': 2},
                    tmin=tmin,
                    tmax=tmax,
                    baseline=None,  # No baseline correction, matching your 2a code
                    preload=True,
                    verbose=False
                )
                
                subj_epochs_list.append(epochs)
            
            # Skip subject if no sessions were processed
            if not subj_epochs_list:
                print(f"No valid sessions found for subject {subj}, skipping.")
                continue
            
            # Concatenate epochs across sessions for this subject
            if len(subj_epochs_list) > 1:
                subj_epochs = mne.concatenate_epochs(subj_epochs_list, verbose=False)
            else:
                subj_epochs = subj_epochs_list[0]
            
            # Get the epoch data
            subj_data = subj_epochs.get_data()
            
            # Apply causal bandpass filter to each trial and channel (matching Dataset 2a)
            n_trials, n_channels, n_times = subj_data.shape
            fs = raw.info['sfreq']  # Sampling frequency (250 Hz for Dataset 2b)
            subj_filtered_data = np.empty_like(subj_data)
            
            if freq_range is None:
                subj_filtered_data = subj_data.copy()
            else:
                lowcut, highcut = freq_range
                for trial in range(n_trials):
                    for ch in range(n_channels):
                        subj_filtered_data[trial, ch, :] = causal_bandpass_filter(
                            subj_data[trial, ch, :],
                            lowcut=lowcut,    # Lower bound of sensorimotor rhythm.
                            highcut=highcut,  # Upper bound of sensorimotor rhythm.
                            fs=fs,
                            order=50     # Lower order for a smoother causal filter.
                        )
            
            # Append processed data, labels, and metadata
            eval_active_X.append(subj_filtered_data)
            eval_active_y.append(subj_epochs.events[:, 2])  # Labels in third column (1 or 2)
            eval_active_metadata.append(subj_epochs.events)
            
            # Print shape to verify
            print(f"Subject {subj}: Epoch data shape {subj_filtered_data.shape}")

        print(f"Loaded data for {len(train_active_X)} subjects.")
        
        train_active_y = encode_labels(train_active_y)
        eval_active_y = encode_labels(eval_active_y)

        align_per_class = 12
        n_subjects = 9
        if(n_classes==2):
            classes = [0, 1]
        else:
            classes = [0, 1, 2, 3]  # Two classes as per your setup
        epsilon = 1e-6
        n_channels = 3
        accuracies = []

        for subj_idx in range(len(train_active_X)):
            # print(subj_idx)
            # Split data into train/test using leave-one-subject-out
            X_target = eval_active_X[subj_idx]
            y_target = eval_active_y[subj_idx]
            
            # Concatenate data from other subjects
            X_source = train_active_X[subj_idx]
            y_source = train_active_y[subj_idx]

            cov_estimator = Covariances(estimator='scm') 
            X_source = cov_estimator.fit_transform(X_source) 
            X_target = cov_estimator.fit_transform(X_target) 

            # Split target data into alignment and test sets
            align_indices = []
            test_indices = []
            for cls in classes:
                cls_indices = np.where(y_target == cls)[0]
                np.random.shuffle(cls_indices)
                align_indices.extend(cls_indices[:align_per_class])
                test_indices.extend(cls_indices[align_per_class:])

            M_source = mean_riemann(X_source)
            M_target_align = mean_riemann(X_target[align_indices])

            M_source_inv_half = np.linalg.inv(scipy.linalg.sqrtm(M_source))
            source_rct = [M_source_inv_half @ C @ M_source_inv_half for C in X_source]
            
            M_target_inv_half = np.linalg.inv(scipy.linalg.sqrtm(M_target_align))
            target_rct = [M_target_inv_half @ C @ M_target_inv_half for C in X_target]
            

            # Compute dispersion d for source
            print("Calculating source dispersions")
            d = 0
            for C_ret in source_rct:
                A_inv_half = np.linalg.inv(scipy.linalg.sqrtm(np.eye(n_channels)))
                C = np.dot(np.dot(A_inv_half, C_ret), A_inv_half)
                log_C = scipy.linalg.logm(C)
                d += np.linalg.norm(log_C, 'fro')**2

            # Compute dispersion tilde_d for T_l
            print("Calculating target dispersions")
            target_align_rct = [target_rct[i] for i in align_indices]
            tilde_d = 0
            for C_ret in target_align_rct:
                A_inv_half = np.linalg.inv(scipy.linalg.sqrtm(np.eye(n_channels)))
                C = np.dot(np.dot(A_inv_half, C_ret), A_inv_half)
                log_C = scipy.linalg.logm(C)
                tilde_d += np.linalg.norm(log_C, 'fro')**2

            # Compute scaling factor s
            s = np.sqrt(d / tilde_d)

            # Stretch target matrices
            target_str = [scipy.linalg.fractional_matrix_power(C_ret, s) for C_ret in target_rct]

            # Compute class means for source
            print("Calculating source class means")
            M_k = []
            for cls in classes:
                source_cls_rct = np.stack([source_rct[i] for i in range(len(y_source)) if y_source[i] == cls])
                source_mean_cls = mean_riemann(source_cls_rct, tol=1e-6, maxiter=100)
                M_k.append(source_mean_cls)

            # Compute class means for labeled target
            print("Calculating labelled target class means")
            tilde_M_k = []
            for cls in classes:
                align_cls_str = np.stack([target_str[i] for i in range(len(y_target[align_indices])) if y_target[align_indices][i] == cls])
                align_mean_cls = mean_riemann(align_cls_str, tol=1e-6, maxiter=100)
                tilde_M_k.append(align_mean_cls)

            # Optimize for U (rotation matrix)
            # Parameterize U = exp(A) where A is skew-symmetric
            manifold = SpecialOrthogonalGroup(n_channels)

            # Define the cost function
            @pymanopt.function.autograd(manifold)
            def cost(U):
                total = 0.0
                for k in range(len(classes)):
                    tilde_M_k_inv_sqrt = linalg.inv(linalg.sqrtm(tilde_M_k[k]))
                    transformed = anp.dot(anp.dot(U, M_k[k]), U.T)
                    arg_logm = anp.dot(anp.dot(tilde_M_k_inv_sqrt, transformed), tilde_M_k_inv_sqrt)
                    log_term = logm_approx(arg_logm)
                    total += frobenius_norm(log_term) ** 2
                return total

            # Create the optimization problem
            problem = Problem(manifold=manifold, cost=cost)

            # Choose a solver and run it
            solver = SteepestDescent()
            U_opt = solver.run(problem)

            # Rotate target matrices
            U_opt = np.array(U_opt.point)
            target_rot = [np.dot(np.dot(U_opt.T, C_str), U_opt) for C_str in target_str]


            X_train = np.concatenate((np.array(source_rct), np.array([target_rot[i] for i in align_indices])))
            y_train = np.concatenate((y_source, y_target[align_indices]))

            mdm = MDM(metric='riemann')
            mdm.fit(X_train, y_train)

            X_test = np.array([target_rot[i] for i in test_indices])
            y_pred = mdm.predict(X_test)

            # Compute and print accuracy
            accuracy = accuracy_score(y_target[test_indices], y_pred)
            accuracies.append(accuracy)
        print(proc_name)
        for subj_idx in range(len(train_active_X)):
            print(f"Subject {subj_idx+1} Test Accuracy: {accuracies[subj_idx]:.2f}")

        print(f"\nMean Cross-Validation Accuracy: {np.mean(accuracies):.2f} ± {np.std(accuracies):.2f}")
        

        
    

In [9]:
twofour_crosssession(2)

/tmp/ipykernel_21007/538256313.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B01: Epoch data shape (120, 3, 751)


/tmp/ipykernel_21007/538256313.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B02: Epoch data shape (120, 3, 751)


/tmp/ipykernel_21007/538256313.py:27: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


...

['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B07: Epoch data shape (120, 3, 751)


/tmp/ipykernel_21007/538256313.py:119: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B08: Epoch data shape (120, 3, 751)


/tmp/ipykernel_21007/538256313.py:119: RuntimeWarning: Highpass cutoff frequency 100.0 is greater than lowpass cutoff frequency 0.5, setting values to 0 and Nyquist.
  raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False)


['EEG:C3', 'EEG:Cz', 'EEG:C4', 'EOG:ch01', 'EOG:ch02', 'EOG:ch03']
Subject B09: Epoch data shape (120, 3, 751)
Loaded data for 9 subjects.
Calculating source dispersions
Calculating target dispersions
Calculating source class means
Calculating labelled target class means
Optimizing...
Iteration    Cost                       Gradient norm     
---------    -----------------------    --------------    
   1         +8.7902785590834343e-01    4.74005640e-02    
   2         +8.4839556949150974e-01    2.75721936e-02    
   3         +8.4575351718636549e-01    2.66346925e-02    
   4         +8.3706445361867765e-01    1.73665888e-02    
   5         +8.3589764609881867e-01    1.51720462e-02    
   6         +8.3296608998667954e-01    4.14292255e-03    
   7         +8.3287996147471310e-01    4.38223268e-03    
   8         +8.3275741895836031e-01    2.72522801e-03    
   9         +8.3271478300070156e-01    1.71324101e-03    
  10         +8.3270999034862536e-01    2.67939114e-03    
  11  